In [5]:
# ------------------------------------------------------------
# 1️⃣  Set-up – install & import
# ------------------------------------------------------------
# !pip install google-cloud-spanner pandas --quiet   # run once

import json, os, pandas as pd
from google.cloud import spanner

PROJECT_ID   = "adg-delivery-moniepoint"
INSTANCE_ID  = "doc-instance"         # <— change
DATABASE_ID  = "utility_docs"         # <— change
# if you need an explicit service-account key:
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/key.json"

spanner_client = spanner.Client(project=PROJECT_ID)
instance = spanner_client.instance(INSTANCE_ID)
database = instance.database(DATABASE_ID)

# ------------------------------------------------------------
# 2️⃣  Helper to run SQL and return a DataFrame
# ------------------------------------------------------------
def spanner_df(sql: str, params: dict | None = None) -> pd.DataFrame:
    with database.snapshot() as snap:
        it    = snap.execute_sql(sql, params=params or {})
        rows  = list(it)                       # materialise first
        if not rows:
            return pd.DataFrame()

        # Now metadata is populated
        cols = [f.name for f in it.metadata.row_type.fields]
        return pd.DataFrame(rows, columns=cols)


# ------------------------------------------------------------
# 3️⃣  Fetch the two tables
# ------------------------------------------------------------
outcomes_raw = spanner_df("SELECT * FROM extracted_fields")
steps_raw    = spanner_df("SELECT * FROM gemini_entities")

from IPython.display import display

pd.set_option("display.max_rows", 200)      # tweak as you like
pd.set_option("display.max_columns", None)  # show every column

print("📄 extracted_fields (first 10 rows)")
display(outcomes_raw.head(10))              # JupyterLab & VS Code render a scrollable grid

print("\n📄 gemini_entities (first 10 rows)")
display(steps_raw.head(10))
print(outcomes_raw.head())

# ------------------------------------------------------------
# 4️⃣  Inspect JSON columns and expand them
# ------------------------------------------------------------
def expand_json(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    """
    For a JSON/STRING column `col`, create one new column per key (prefix.key)
    and return a new DataFrame with those columns appended.
    """
    def _to_dict(x):
        if x is None:
            return {}
        if isinstance(x, dict):
            return x
        try:
            return json.loads(x)
        except Exception:
            return {}

    expanded = (
        df[col]
        .apply(_to_dict)
        .apply(pd.Series)
        .add_prefix(f"{prefix}.")
    )
    return pd.concat([df.drop(columns=[col]), expanded], axis=1)

# # Example: the outcomes table stores a JSON string in `heuristic_result`
# if "heuristic_result" in outcomes_raw.columns:
#     outcomes = expand_json(outcomes_raw, "heuristic_result", "heuristic")
# else:
#     outcomes = outcomes_raw.copy()
#
# # Example: the steps table stores per-entity extraction in a JSON column
# if "entities_json" in steps_raw.columns:
#     steps = expand_json(steps_raw, "entities_json", "ent")
# else:
#     steps = steps_raw.copy()

# ------------------------------------------------------------
# 5️⃣  Quick glance at what we have
# ------------------------------------------------------------



📄 extracted_fields (first 10 rows)


,gcs_uri,supply_address_label,supply_address_value,supply_address_conf,name_label,name_value,name_conf,period_label,period_value,period_conf,meter_number_label,meter_number_value,meter_number_conf,bill_month_label,bill_month_value,bill_month_conf,customer_account_label,customer_account_value,customer_account_conf,provider_acronym_label,provider_acronym_value,provider_acronym_conf,service_address_label,service_address_value,service_address_conf,meter_type_label,meter_type_value,meter_type_conf,transaction_date_label,transaction_date_value,transaction_date_conf,address_label,address_value,address_conf
0,gs://adg-delivery-moniepoint-docs-bucket-001/t...,Supply Address:,"MYPA ROAD, BOSSO Und St. Bosso",0.90,Name:,UMAR MUSA,0.97,Period:,02/10/2024-01/11/2024,0.95,Meter Number:,0,0.51,,,NaN,Account Number:,712981471,0.99,,,None,,,NaN,,,NaN,,,NaN,Supply Address:,"MYPA ROAD, BOSSO Und St. Bosso",0.90
1,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Customer Name,GRABA ISHARK,0.99,,,NaN,Meter Number,45060179160,0.99,,,NaN,,,NaN,,,None,Service Address,"GWARRI VILLAGE MPAPE,, MPAPE",0.98,Meter Type,Prepaid,0.99,Transaction Date,action Date,0.98,Service Address,"GWARRI VILLAGE MPAPE,, MPAPE",0.98
2,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Account Name:,SULEIMAN SAMINU,0.98,,,NaN,Meter #:,,0.92,Bill Month:,99845359,0.95,Customer Account #: 1353046,1353046,0.96,,,None,,,NaN,,,NaN,,,NaN,Address:,SULEIMAN SAMINU,0.98
3,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,DT Name:,"June, 2024",0.97,,,NaN,Meter #:,,0.80,Bill Month:,"June, 2024",0.97,Customer Account #:,Bill #: 18521345,0.94,,,None,,,NaN,,,NaN,,,NaN,Address:,Numan,0.98
4,gs://adg-delivery-moniepoint-docs-bucket-001/t...,Supply Address:,"26,OGBOGORO ROAD",0.92,Name,82312662eous Bal: N,0.69,,,NaN,Meter,0,0.54,,,NaN,Account Number:,82312662eous Bal: N,0.69,,,None,,,NaN,,,NaN,,,NaN,Supply Address:,"26,OGBOGORO ROAD",0.92
5,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Account Name,MRS EGO MBA,0.97,,,NaN,Meter Number,0124001115953,0.98,,,NaN,Account Name,N100.00,0.94,,,None,,,NaN,,,NaN,,,NaN,Address,Successful,0.91
6,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Account Name REDCO,e REDCO,0.66,,,NaN,Meter and Account Details,,0.95,,,NaN,Meter and Account Details,,0.95,,,None,,,NaN,,,NaN,,,NaN,,,NaN
7,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Markerter Name=,Due Date 13/12/2024,0.98,,,NaN,MeterNo =,,0.88,,,NaN,Account No 10/81/76/0064-01,CIN: 030927205161601100615002,0.88,,,None,,,NaN,,,NaN,,,NaN,,,NaN
8,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Account Name,01325705695,0.98,,,NaN,Meter Number,01325705695,0.98,,,NaN,Account Name,01325705695,0.98,,,None,,,NaN,,,NaN,,,NaN,Address,44 APENA STREET Lawanson,0.95
9,gs://adg-delivery-moniepoint-docs-bucket-001/t...,,,NaN,Name: MRS. FOWOBADE DEBORAH,Previ,0.90,,,NaN,MeterNo:,,0.98,,,NaN,OldAccountNo:,DUE DATE: 12/16/2024,0.99,,,None,,,NaN,,,NaN,,,NaN,"S/Address: 11, OTUNAGBAKIN ONIYA LABI","11, OTUNAGBAKIN ONIYA LABI",0.98



📄 gemini_entities (first 10 rows)


,gcs_uri,page,entity,value,confidence
0,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Account Number,712981471,0.95
1,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Avg. Daily Consumption (kWh),0,0.95
2,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Bill Month,Nov,0.95
3,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Contract Number,20129860201,0.95
4,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Customer Account,72981471,0.95
5,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Customer Name,UMAR MUSA,0.95
6,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Feeder,MINNA ZARUMAI_ZARUMAI_HAJJ CAMP,0.95
7,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Meter Number,0,0.95
8,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Old Account No,93-8103-5168-01-621,0.95
9,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Period,02/10/2024-01/11/2024,0.95


                                             gcs_uri supply_address_label  \
0  gs://adg-delivery-moniepoint-docs-bucket-001/t...      Supply Address:   
1  gs://adg-delivery-moniepoint-docs-bucket-001/t...                        
2  gs://adg-delivery-moniepoint-docs-bucket-001/t...                        
3  gs://adg-delivery-moniepoint-docs-bucket-001/t...                        
4  gs://adg-delivery-moniepoint-docs-bucket-001/t...      Supply Address:   

             supply_address_value  supply_address_conf     name_label  \
0  MYPA ROAD, BOSSO Und St. Bosso                 0.90          Name:   
1                                                  NaN  Customer Name   
2                                                  NaN  Account Name:   
3                                                  NaN       DT Name:   
4                26,OGBOGORO ROAD                 0.92           Name   

            name_value  name_conf period_label           period_value  \
0            UMAR MUSA   

In [4]:
# ------------------------------------------------------------
# 6️⃣  Synonym-aware merge  +  explode multi-value columns
# ------------------------------------------------------------
import re, json, pandas as pd
from IPython.display import display

# 6.1 ─ entity-synonym map  (left = Gemini, right = canonical key)
synonyms = {
    "account number":      "customer_account",
    "old account no":      "customer_account",
    "customer account":    "customer_account",
    "customer name":       "name",
    "name":                "name",
    "period":              "period",
    "bill month":          "bill_month",
    "meter number":        "meter_number",
    "meter #":             "meter_number",
    "meter type":          "meter_type",
    "avg. daily consumption (kwh)": None,          # not tracked in extracted_fields
    "contract number":     None,
    "feeder":              None,
    "provideracronym":     "provider_acronym",
    "supply address":      "supply_address",
    "service address":     "service_address",
    "address":             "address",
    "transaction date":    "transaction_date",
}

def canon(s: str) -> str:
    """lower, strip punctuation/spaces."""
    return re.sub(r"_+", "_",
                  re.sub(r"[^0-9a-z]+", "_", s.lower())).strip("_")

# --- 6.2 normalise entity names & drop ones we don't track ----
gem = (
    steps_raw
    .assign(key=lambda d: d["entity"].astype(str).apply(lambda e: synonyms.get(canon(e))))
    .dropna(subset=["key"])                              # ditch un-mapped entities
    .rename(columns={"value": "gem_value",
                     "confidence": "gem_conf"})
)

# --- 6.3  build a long (“tidy”) view of extracted_fields -----
def to_rows(df, prefix):
    """Yield (gcs_uri, key, label, value, conf) one per candidate value."""
    lbl = df.get(f"{prefix}_label")
    val = df.get(f"{prefix}_value")
    cof = df.get(f"{prefix}_conf")

    for uri, L, V, C in zip(df["gcs_uri"], lbl, val, cof):
        # split / parse multi-value cells ----------------------
        for l, v, c in zip(
                *[parse_multi(col) for col in (L, V, C)]
        ):
            yield uri, prefix, l, v, c

def parse_multi(x):
    """
    Return a list of candidate pieces from a cell:
    – JSON list  → list
    – "a; b; c"  → ['a','b','c']
    – "a\nb"     → ['a','b']
    – scalar     → [scalar]
    """
    if x is None or x == "":
        return [None]
    if isinstance(x, (list, tuple)):
        return list(x)
    try:
        j = json.loads(x)
        if isinstance(j, list):
            return j
    except Exception:
        pass
    # split on ; or newline
    if isinstance(x, str) and re.search(r"[;\n]", x):
        return [p.strip() for p in re.split(r"[;\n]", x) if p.strip()]
    return [x]

long_rows = []
for prefix in {c.rsplit("_value", 1)[0] for c in outcomes_raw.columns if c.endswith("_value")}:
    long_rows.extend(to_rows(outcomes_raw, prefix))

extracted_tidy = pd.DataFrame(long_rows,
                              columns=["gcs_uri", "key",
                                       "ext_label", "ext_value", "ext_conf"])

# --- 6.4  merge & show ---------------------------------------
merged = (
    gem
    .merge(extracted_tidy, on=["gcs_uri", "key"], how="left")
    .sort_values(["gcs_uri", "page", "entity"])
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

print(f"🔎 merged rows: {len(merged):,}")
display(merged.head(30))


🔎 merged rows: 27


,gcs_uri,page,entity,gem_value,gem_conf,key,ext_label,ext_value,ext_conf
0,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Period,02/10/2024-01/11/2024,0.95,period,Period:,02/10/2024-01/11/2024,0.95
1,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,providerAcronym,AEDC,0.95,provider_acronym,None,None,NaN
2,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Address,FLAT NO 8 WAPPAH,0.95,address,Address:,SULEIMAN SAMINU,0.98
3,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Period,"December, 2024",0.95,period,None,None,NaN
4,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,providerAcronym,ke,0.95,provider_acronym,None,None,NaN
5,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Address,LAYIN AUDU MAI KARFE,0.90,address,Address:,Numan,0.98
6,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Period,"June, 2024",0.90,period,None,None,NaN
7,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,providerAcronym,YEDC,0.90,provider_acronym,None,None,NaN
8,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Address,"26,OGBOGORO ROAD RUMUOLUMENI",0.95,address,Supply Address:,"26,OGBOGORO ROAD",0.92
9,gs://adg-delivery-moniepoint-docs-bucket-001/t...,1,Period,DEC-2024,0.95,period,None,None,NaN


In [9]:
print("👀 unique entity values from gemini_entities:")
display(steps_raw["entity"].dropna().unique()[:50])   # first 50 unique strings

print("\n👀 first few column prefixes in extracted_fields (everything before _value):")
prefixes = sorted({c.rsplit("_value", 1)[0] for c in outcomes_raw.columns if c.endswith("_value")})
display(prefixes[:50])


👀 unique entity values from gemini_entities:


array(['Account Number', 'Avg. Daily Consumption (kWh)', 'Bill Month',
       'Contract Number', 'Customer Account', 'Customer Name', 'Feeder',
       'Meter Number', 'Old Account No', 'Period', 'Supply Address',
       'Transaction Date', 'providerAcronym', 'Meter Type',
       'Service Address', 'Address', 'Meter #'], dtype=object)


👀 first few column prefixes in extracted_fields (everything before _value):


['address',
 'bill_month',
 'customer_account',
 'meter_number',
 'meter_type',
 'name',
 'period',
 'provider_acronym',
 'service_address',
 'supply_address',
 'transaction_date']